# 🧬 **Projeto SIEST: Sistema de Inteligência Epidemiológica e Socioespacial**
## Notebook 04: Extração de Dados Socioespaciais e Saneamento (Camada Bronze)

Este notebook tem como objetivo extrair as matrizes geográficas (polígonos) que representam a vulnerabilidade urbana do município de Campinas, focando no **ODS 6 (Água Potável e Saneamento)**. O intuito é cruzar estas áreas críticas com os focos de doenças hídricas e arboviroses.

**Estratégia Arquitetural (Ingestão Local):** Devido a bloqueios de *Firewall* (Timeouts) e indisponibilidade de granularidade fina nas APIs federais (IBGE/SIDRA), a arquitetura foi adaptada para consumir metadados vetoriais diretos (Shapefiles) do banco PostGIS da Prefeitura de Campinas.

**Fontes de Dados:** * Web GIS Portal Geoambiental (Prefeitura de Campinas).
* Camadas: Núcleos Urbanos (SEHAB) e Risco de Inundação (CPRM/IPT).

In [ ]:
!pip install geopandas fiona shapely -q

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import warnings

# Ocultar avisos de depreciação de bibliotecas internas
warnings.filterwarnings('ignore')

PASTA_BRONZE = 'dados_saneamento_parquet'
os.makedirs(PASTA_BRONZE, exist_ok=True)

# Mapeamento dos ficheiros Shapefile brutos transferidos para a máquina virtual
# (É imperativo que os ficheiros .dbf, .shx e .prj estejam no mesmo diretório)
ARQUIVOS_LOCAIS = {
    'vulnerabilidade_habitacional': 'vulnerabilidade.shp',
    'risco_inundacao': 'inundacao.shp'
}

### 1. Motor de Transformação e Carga (ETL Vetorial)

Este bloco itera sobre os ficheiros locais brutos, utiliza o `GeoPandas` para extrair a geometria e as tabelas de atributos, padroniza o Sistema de Referência de Coordenadas (CRS) para o padrão global GPS (EPSG:4326) e persiste os dados na Camada Bronze utilizando o formato colunar `.parquet` com compressão `snappy`.

In [ ]:
print("A iniciar processamento vetorial dos ficheiros SHP locais...\n")

for nome_camada, nome_arquivo in ARQUIVOS_LOCAIS.items():
    print(f"⚙️ A processar geometria espacial de {nome_camada.upper()}...")

    try:
        # Verifica integridade do ficheiro local
        if not os.path.exists(nome_arquivo):
            print(f"⚠️ ATENÇÃO: O ficheiro '{nome_arquivo}' não foi encontrado. Falha na ingestão.\n")
            continue

        # Leitura nativa do Shapefile
        gdf_camada = gpd.read_file(nome_arquivo)

        # Padronização do CRS (Sistema de Coordenadas)
        # Origem: SIRGAS 2000 UTM 23S (EPSG:31983) -> Destino: Lat/Long WGS84 (EPSG:4326)
        if gdf_camada.crs is None:
            gdf_camada.set_crs(epsg=31983, inplace=True)
        gdf_camada = gdf_camada.to_crs(epsg=4326)

        # Carga (Save) em formato Parquet para a Camada Bronze
        caminho_saida = f"{PASTA_BRONZE}/{nome_camada}_campinas.parquet"
        gdf_camada.to_parquet(caminho_saida, index=False, compression='snappy')

        print(f"✅ Sucesso! Camada guardada com {len(gdf_camada)} polígonos em: {caminho_saida}\n")

    except Exception as e:
        print(f"❌ Erro crítico ao processar a camada {nome_camada}: {e}\n")

print("🚀 PIPELINE SOCIOESPACIAL CONCLUÍDO COM EXCELÊNCIA!")

### 2. Auditoria e Validação Visual (Camada Bronze)

Rotina de segurança para garantir a integridade dos dados guardados no Data Lake. O script lê os ficheiros `.parquet` recém-gerados e realiza a renderização espacial imediata (Plot) para comprovar a conversão correta das coordenadas geográficas.

In [ ]:
caminho_vuln = f"{PASTA_BRONZE}/vulnerabilidade_habitacional_campinas.parquet"
caminho_inund = f"{PASTA_BRONZE}/risco_inundacao_campinas.parquet"

gdf_vuln = gpd.read_parquet(caminho_vuln)
gdf_inund = gpd.read_parquet(caminho_inund)

print("📊 --- VALIDAÇÃO: VULNERABILIDADE HABITACIONAL ---")
print(f"Total de Linhas (Polígonos): {len(gdf_vuln)}")
display(gdf_vuln[['NOME_AREA', 'geometry']].head(3))

print("\n🌊 --- VALIDAÇÃO: RISCO DE INUNDAÇÃO ---")
print(f"Total de Linhas (Polígonos): {len(gdf_inund)}")
display(gdf_inund[['CLASSE', 'geometry']].head(3))

# ---------------------------------------------------------
# 4. VALIDAÇÃO VISUAL (MAPAS RÁPIDOS)
# ---------------------------------------------------------
print("\n🗺️ A gerar mapas de validação espacial...")
fig, eixos = plt.subplots(1, 2, figsize=(16, 8))

# Camada 1: Vulnerabilidade
gdf_vuln.plot(ax=eixos[0], color='darkred')
eixos[0].set_title('Polígonos de Vulnerabilidade Habitacional\n(Assentamentos/Núcleos Informais)', fontsize=12)
eixos[0].set_axis_off()

# Camada 2: Inundação
gdf_inund.plot(ax=eixos[1], color='darkblue')
eixos[1].set_title('Polígonos de Risco de Inundação\n(Áreas Críticas de Alagamento)', fontsize=12)
eixos[1].set_axis_off()

plt.tight_layout()
plt.show()